# Clinical phenotype analysis

This tutorial continues the HBC1 Step 2 phenotype workflow. PhenoMap first compares the top phenotype-associated cancer cells against the remaining cancer cells, then carries the resulting genes into the TCGA-BRCA reference cohort to derive a compact clinical signature. The frozen signature is scored in external cohorts and evaluated with median-split Kaplan-Meier curves.

Heavy discovery and validation scripts are kept as runnable entry points. The notebook displays the final HBC1 outputs generated by the existing analysis code so the figures match the paper-style outputs.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
from IPython.display import HTML, IFrame, display


def find_repo_root():
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "README.md").exists() and (path / "step1").exists():
            return path
    raise RuntimeError(
        "Please run this notebook from inside the PhenoMap repository."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from step2.utils.survival_signature import (
    compute_signature_risk,
    evaluate_signature_km,
    fit_lifelines_l1_cox_signature,
    plot_signature_km,
)

DATA_ROOT = Path(os.environ["PHENOMAP_DATA"]).expanduser().resolve()
HBC1_DIR = DATA_ROOT / "HBC1"
PHENOTYPE_DIR = HBC1_DIR / "phenotype"
OUTPUT_DIR = REPO_ROOT / "outputs" / "04_clinical_phenotype_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEA_PATH = PHENOTYPE_DIR / "DEA_results_best.csv"
SIGNATURE_CANDIDATE_PATH = PHENOTYPE_DIR / "up_genes.txt"
TCGA_EXPR_PATH = DATA_ROOT / "TCGA" / "expr_aligned_plus.tsv"
TCGA_CLIN_PATH = DATA_ROOT / "TCGA" / "clinical_aligned_plus.tsv"
SIGNATURE_PATH = OUTPUT_DIR / "hbc1_l1cox_signature_genes.csv"
SIGNATURE_DIAGNOSTICS_PATH = OUTPUT_DIR / "hbc1_l1cox_penalizer_diagnostics.csv"

RUN_HEAVY_STEPS = False

required = [DEA_PATH, TCGA_EXPR_PATH, TCGA_CLIN_PATH]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("\n".join(missing))

print(f"Repository:       {REPO_ROOT}")
print(f"HBC1 phenotype:   {PHENOTYPE_DIR}")
print(f"Tutorial outputs: {OUTPUT_DIR}")
print(f"Run heavy steps:  {RUN_HEAVY_STEPS}")


## 1. Differential expression in high-score cancer cells

The discovery step restricts to cancer epithelial cells, compares the highest phenotype-score cells against the remaining cancer cells, and exports the ranked DEA table used by the downstream clinical signature workflow.


In [2]:
DEA_SCRIPT = Path(os.environ["PHENOMAP_DEA_SCRIPT"]).expanduser().resolve()
dea_cmd = ["python", str(DEA_SCRIPT)]
print(" ".join(dea_cmd))

if RUN_HEAVY_STEPS:
    subprocess.run(dea_cmd, cwd=REPO_ROOT, check=True)


python analysis/projects/phenotype/325/get_gene.py


In [3]:
dea = pd.read_csv(DEA_PATH)
dea_summary = (
    dea.groupby('Significance')
    .size()
    .rename('n_genes')
    .reset_index()
    .sort_values('Significance')
)
display(dea_summary)
dea.head(12)

,Significance,n_genes
0,Down,22
1,Not Sig,194
2,Up,96


,names,scores,logfoldchanges,pvals,pvals_adj,minuslog10padj,plot_y,Significance,abs_score
0,FASN,41.731415,1.490987,0.000000e+00,0.000000e+00,300.000000,261.200327,Up,41.731415
1,ABCC11,31.254158,2.200826,1.960259e-214,2.045204e-212,211.689263,211.689263,Up,31.254158
2,EPCAM,29.062159,0.687095,1.080351e-185,6.763000e-184,183.169861,183.169861,Up,29.062159
3,SERHL2,28.021671,1.962471,8.847815e-173,4.615610e-171,170.335771,170.335771,Up,28.021671
4,SCD,27.704683,1.039785,6.131548e-169,2.741678e-167,166.561984,166.561984,Up,27.704683
5,FOXA1,26.141714,0.743043,1.224336e-150,4.790213e-149,148.319645,148.319645,Up,26.141714
6,TENT5C,26.052164,2.028183,1.271499e-149,4.421991e-148,147.354382,147.354382,Up,26.052164
7,KRT7,24.307564,0.466830,1.630599e-130,5.103776e-129,128.292108,128.292108,Not Sig,24.307564
8,CENPF,24.177284,1.987978,3.857628e-129,1.097670e-127,126.959528,126.959528,Up,24.177284
9,HMGA1,23.276913,1.783607,7.597287e-120,1.981626e-118,117.702978,117.702978,Up,23.276913


In [4]:
import matplotlib.pyplot as plt

dea_pdf = OUTPUT_DIR / 'Volcano_top5.pdf'
plot_dea = dea.copy()
colors = plot_dea['Significance'].map({'Up': '#E64B35', 'Down': '#4DBBD5'}).fillna('#7f7f7f')
fig, ax = plt.subplots(figsize=(7.2, 6.0), dpi=160)
ax.scatter(plot_dea['logfoldchanges'], plot_dea['minuslog10padj'], c=colors, s=18, alpha=0.82, linewidths=0)
for _, row in plot_dea.sort_values('abs_score', ascending=False).head(10).iterrows():
    ax.text(row['logfoldchanges'], row['minuslog10padj'], row['names'], fontsize=8)
ax.axvline(0.5, color='0.5', linestyle=':', linewidth=1)
ax.axvline(-0.5, color='0.5', linestyle=':', linewidth=1)
ax.set_xlabel('log2 Fold Change')
ax.set_ylabel('-log10 adjusted P-value')
ax.set_title('Top phenotype-associated cancer-cell DEA')
ax.spines[['top', 'right']].set_visible(False)
fig.savefig(dea_pdf, bbox_inches='tight', transparent=True)
plt.close(fig)
display(IFrame(src='outputs/Volcano_top5.pdf', width='100%', height=620))

## 2. Clinical signature from TCGA-BRCA

The DEA genes are carried into the TCGA-BRCA reference data used for Step 2 training. The existing analysis code performs sparse clinical feature selection and stores the signed/weighted signature used by validation. The compact display below shows the top 20 weighted genes from the frozen signature table.

In [5]:
signature_entry_points = [
    "step2/utils/survival_signature.py",
    str(DATA_ROOT / "analysis" / "step2_km_external_from_signature.py"),
    str(DATA_ROOT / "analysis" / "step2_survival_analysis.py"),
]
for entry in signature_entry_points:
    print(entry)


In [6]:
candidate_genes = (
    dea.loc[dea['Significance'].isin(['Up', 'Down'])]
    .sort_values('abs_score', ascending=False)['names']
    .dropna()
    .drop_duplicates()
    .tolist()
)

tcga_expr = pd.read_csv(TCGA_EXPR_PATH, sep='\t', index_col=0)
tcga_clin = pd.read_csv(TCGA_CLIN_PATH, sep='\t', index_col=0)

signature, diagnostics = fit_lifelines_l1_cox_signature(
    tcga_expr,
    tcga_clin,
    candidate_genes,
    time_col='OS.time',
    event_col='OS.event',
    min_genes=16,
    max_genes=20,
    target_genes=20,
)
signature.to_csv(SIGNATURE_PATH, index=False)
diagnostics.to_csv(SIGNATURE_DIAGNOSTICS_PATH, index=False)

print(f'DEA candidates entering L1-Cox: {len(candidate_genes)}')
print(f'L1-Cox signature genes: {len(signature)}')
display(diagnostics[diagnostics['selected']].reset_index(drop=True))
signature

DEA candidates entering L1-Cox: 118
L1-Cox signature genes: 20


,penalizer,n_nonzero,error,selected,selection_mode
0,0.01475,20,,True,target_sparsity


,gene,coef
0,SERPINA3,-0.203251
1,RAPGEF3,0.110934
2,EGFL7,0.099530
3,MZB1,-0.089946
4,PTRHD1,-0.087549
5,ADAM9,0.072463
6,FOXP3,-0.051203
7,TENT5C,-0.045044
8,EIF4EBP1,0.044850
9,LGALSL,0.038397


## 3. External cohort KM validation

The validation step starts from each external cohort's expression and clinical tables. The notebook computes signed ssGSEA risk scores from the frozen signature, applies a median split, writes a fresh `*_clinical_scores.tsv`, and generates a fresh KM PDF using the established PhenoMap phenotype plotting style.

In [7]:
external_cohorts = [
    {
        'label': 'GSE103091 OS',
        'expr': DATA_ROOT / 'GSE103091' / 'expr.tsv',
        'clinical': DATA_ROOT / 'GSE103091' / 'clinical.tsv',
        'scores': 'GSE103091_OS_clinical_scores.tsv',
        'pdf': 'GSE103091_OS.pdf',
        'time': 'OS.time',
        'event': 'OS.event',
        'ylabel': 'Overall Survival',
    },
    {
        'label': 'GSE103091 MFS',
        'expr': DATA_ROOT / 'GSE103091' / 'expr.tsv',
        'clinical': DATA_ROOT / 'GSE103091' / 'clinical.tsv',
        'scores': 'GSE103091_MFS_clinical_scores.tsv',
        'pdf': 'GSE103091_MFS.pdf',
        'time': 'MFS.time',
        'event': 'MFS.event',
        'ylabel': 'Metastasis-Free Survival',
    },
    {
        'label': 'EGAS50000000475 / TNBC_ST OS',
        'expr': DATA_ROOT / 'TNBC_ST' / 'Processed_Clinical' / 'expr.tsv',
        'clinical': DATA_ROOT / 'TNBC_ST' / 'Processed_Clinical' / 'clinical.tsv',
        'scores': 'TNBC_ST_OS_clinical_scores.tsv',
        'pdf': 'TNBC_ST_OS.pdf',
        'time': 'OS.time',
        'event': 'OS.event',
        'ylabel': 'Overall Survival',
    },
    {
        'label': 'EGAS50000000475 / TNBC_ST DRFS',
        'expr': DATA_ROOT / 'TNBC_ST' / 'Processed_Clinical' / 'expr.tsv',
        'clinical': DATA_ROOT / 'TNBC_ST' / 'Processed_Clinical' / 'clinical.tsv',
        'scores': 'TNBC_ST_DRFS_clinical_scores.tsv',
        'pdf': 'TNBC_ST_DRFS.pdf',
        'time': 'DRFS.time',
        'event': 'DRFS.event',
        'ylabel': 'Distant Relapse-Free Survival',
    },
    {
        'label': 'GSE7390 OS',
        'expr': DATA_ROOT / 'GSE7390' / 'expr.tsv',
        'clinical': DATA_ROOT / 'GSE7390' / 'clinical.tsv',
        'scores': 'GSE7390_OS_clinical_scores.tsv',
        'pdf': 'GSE7390_OS.pdf',
        'time': 'OS.time',
        'event': 'OS.event',
        'ylabel': 'Overall Survival',
    },
    {
        'label': 'GSE7390 MFS',
        'expr': DATA_ROOT / 'GSE7390' / 'expr.tsv',
        'clinical': DATA_ROOT / 'GSE7390' / 'clinical.tsv',
        'scores': 'GSE7390_MFS_clinical_scores.tsv',
        'pdf': 'GSE7390_MFS.pdf',
        'time': 'MFS.time',
        'event': 'MFS.event',
        'ylabel': 'Metastasis-Free Survival',
    },
]

for item in external_cohorts:
    for key in ['expr', 'clinical']:
        if not item[key].exists():
            raise FileNotFoundError(item[key])

pd.DataFrame([{k: str(v) for k, v in item.items()} for item in external_cohorts])

In [8]:
summary_rows = []

for item in external_cohorts:
    expr = pd.read_csv(item['expr'], sep='\t', index_col=0)
    clin = pd.read_csv(item['clinical'], sep='\t', index_col=0)
    expr.columns = expr.columns.astype(str)
    clin.index = clin.index.astype(str)
    clin[item['time']] = pd.to_numeric(clin[item['time']], errors='coerce') / 30.4375
    clin[item['event']] = pd.to_numeric(clin[item['event']], errors='coerce')
    clin = clin.dropna(subset=[item['time'], item['event']])
    risk = compute_signature_risk(
        expr,
        clin,
        signature,
        time_col=item['time'],
        event_col=item['event'],
    )
    risk.to_csv(OUTPUT_DIR / item['scores'], sep='\t')
    stats = evaluate_signature_km(risk, time_col=item['time'], event_col=item['event'], cutoff=150)
    plot_signature_km(
        risk,
        OUTPUT_DIR / item['pdf'],
        title=item['label'],
        time_col=item['time'],
        event_col=item['event'],
        time_label='Time (Months)',
        survival_label=item['ylabel'],
        cutoff=150,
        stats=stats,
    )
    summary_rows.append({
        'cohort': item['label'],
        **stats,
        'signature_genes_available': int(signature['gene'].isin(expr.index).sum()),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / 'external_km_summary.csv', index=False)
summary

,cohort,n,events,p_value,c_index,hazard_ratio,hazard_ratio_ci_low,hazard_ratio_ci_high,signature_genes_available
0,GSE103091 OS,107,29,0.584024,0.508864,1.226529,0.589773,2.550770,19
1,GSE103091 MFS,238,31,0.477993,0.501712,1.293967,0.633508,2.642984,19
2,EGAS50000000475 / TNBC_ST OS,92,19,0.625520,0.522332,1.251511,0.507377,3.087010,20
3,EGAS50000000475 / TNBC_ST DRFS,92,20,0.527078,0.535862,1.328235,0.549527,3.210412,20
4,GSE7390 OS,198,48,0.221247,0.560951,1.427964,0.804404,2.534898,17
5,GSE7390 MFS,198,57,0.209917,0.545505,1.397747,0.826092,2.364986,17


In [9]:
for item in external_cohorts:
    display(HTML(f'<h3>{item["label"]}</h3>'))
    display(IFrame(src=f'outputs/{item["pdf"]}', width='100%', height=620))

## Main outputs

- `DEA_results_best.csv`: cancer-cell phenotype-region DEA table.
- `up_genes.txt`: candidate genes from the high-score phenotype region.
- `Global_Beta_Top_Genes.csv`: frozen signed/weighted clinical signature table.
- `*_clinical_scores.tsv`: external cohort risk scores and median-split groups.
- `*_OS.pdf`, `*_MFS.pdf`, `*_DRFS.pdf`: final existing KM figures embedded above.